# scCODA


### DMG-ATLAS Data

Note: run saveMeta.R first to prepare metadata locations for this analysis

In [1]:
# load packages
# Setup
import importlib
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import pickle as pkl
import matplotlib.pyplot as plt
from sccoda.util import comp_ana as mod
from sccoda.util import cell_composition_data as dat
from sccoda.util import data_visualization as viz
import anndata as ad
import scanpy as sc
import numpy as np


2026-10-20 23:50:43.981480: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-20 23:50:43.994952: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-20 23:50:44.665694: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
# LOAD DMG-ATLAS META
meta = pd.read_csv("../output/meta_atlas_dmg.csv")


In [ ]:
meta["sample_id"].unique()

In [ ]:

meta_ol = meta.copy()
# counts of cell types per sample
counts = pd.crosstab(
    meta_ol["sample_id"],
    meta_ol["Cell_Type"]
)

print(counts.shape)
print(counts.head())
# sample meta
sample_meta = (
    meta_ol[["sample_id", "condition"]]
    .drop_duplicates()
    .set_index("sample_id")
)

sample_meta = sample_meta.loc[counts.index]

cond_order = [
    "control",
    "tumor"
]

sample_meta["condition"] = pd.Categorical(
    sample_meta["condition"],
    categories=cond_order,
    ordered=True
)

sample_meta = sample_meta.sort_values("condition")
counts = counts.loc[sample_meta.index]


# check rows match
all(counts.index == sample_meta.index)
import anndata as ad

count_adata = ad.AnnData(
    X=counts.values.astype(int),
    obs=sample_meta.copy(),
    var=pd.DataFrame(index=counts.columns)
)

count_adata.obs["condition"] = sample_meta["condition"].values
print(count_adata)
proportions = counts.div(
    counts.sum(axis=1),
    axis=0
)

proportions["condition"] = sample_meta["condition"]

print(
    proportions.groupby("condition")
    .mean()
    .T
    .to_string()
)

(23, 19)
Cell_Type   Astrocyte  Early OL  Endothelial  Ependymal  Erythroblast  \
sample_id                                                               
GSM7305260         22       137            5          0             0   
GSM7305265        159        93            0          7             0   
GSM7305266         17       155           28          1             0   
GSM7305270        101       116            2         11             0   
GSM7305276          4        19            0          0             4   

Cell_Type   Fetal COP  Fetal Neuronal Progenitor  \
sample_id                                          
GSM7305260          4                         85   
GSM7305265          0                        349   
GSM7305266         20                        342   
GSM7305270          0                       1040   
GSM7305276          0                         48   

Cell_Type   Intermediate Proliferative Cell  Lymphocyte  Mesenchymal  Myeloid  \
sample_id                        

In [ ]:
# Convert AnnData count matrix to a DataFrame
counts_df = count_adata.to_df()

# Convert all values to numeric; non-numeric values become NaN
counts_df = counts_df.apply(pd.to_numeric, errors="coerce")

# Check for unexpected non-numeric values
if counts_df.isna().any().any():
    print("Warning: non-numeric or missing values found in the count matrix.")
    print("Missing values:", counts_df.isna().sum().sum())

# Stop rather than silently treating missing counts as zero
if counts_df.isna().any().any():
    raise ValueError("Inspect and resolve missing/non-numeric values before filtering.")

# Convert to NumPy array
counts = counts_df.to_numpy(dtype=float)

# Minimum number of samples in which a cell type must be present
min_samples = int(np.ceil(0.20 * counts.shape[0]))

# Number of samples with > 0 cells for each cell type
samples_present = (counts > 0).sum(axis=0)
keep = samples_present >= min_samples

print("Samples required:", min_samples)
print("Cell types retained:", keep.sum(), "of", len(keep))
print("Cell types to review:", counts_df.columns[~keep].tolist())

# Create filtered AnnData object
data_filtered = count_adata[:, keep].copy()

Samples required: 5
Cell types retained: 18 of 19
Cell types to review: ['PAX6+ HOXB3+ Neuroblast']


In [ ]:
model = mod.CompositionalAnalysis(
    data_filtered,
    formula="condition",
    reference_cell_type="Myeloid"
)

In [ ]:
result = model.sample_nuts(
    num_results=5000,
    num_burnin=2000,
    verbose=True
)

In [ ]:
# result table
print(result.summary_extended())

Compositional Analysis summary (extended):

Data: 23 samples, 18 cell types
Reference index: 10
Formula: condition
Spike-and-slab threshold: 0.901

MCMC Sampling: Sampled 5000 chain states (2000 burnin samples) in 928.143 sec. Acceptance rate: 72.7%

Intercepts:
                                 Final Parameter  HDI 3%  HDI 97%     SD  \
Cell Type                                                                  
Astrocyte                                  1.721   1.455    1.994  0.145   
Early OL                                   0.680   0.296    1.052  0.201   
Endothelial                               -0.977  -1.455   -0.530  0.241   
Ependymal                                 -1.574  -1.978   -1.184  0.218   
Erythroblast                              -1.624  -2.078   -1.220  0.230   
Fetal COP                                 -1.570  -2.015   -1.178  0.221   
Fetal Neuronal Progenitor                 -1.665  -2.124   -1.196  0.250   
Intermediate Proliferative Cell           -1.438  -1.

### MS-ATLAS Data

In [2]:
meta = pd.read_csv("../output/meta_atlas_ms.csv")
#meta = meta[meta["dev.group"]]

In [3]:
meta["sample_id"].unique()

array(['SC075_MW_1275_pons_s7', 'SC076_MW_771_pons_s7',
       'SC077_MW_4327_pons_S3', 'SC079_MW_1550_pons_s7',
       'SC080_MW_5309_pons_s7', 'SC081_MW_1864_pons_sec7',
       'SC082_MW_5925_pons_sec7', 'SC084_MW_1860_pons_sec7',
       'SC085_MW_1740_pons_sec7', 'SC087_MW_4722_pons_sec7',
       'SC088_MW_4332_pons_sec9', 'SC089_MW_1499_pons_sec9',
       'SC090_MW_4925_pons_sec7', 'SC126_MW_1136_pons_s5',
       'SC127_MW_1229_pons_s7', 'SC128_MW_1538_pons_s5',
       'SC129_MW_4593_pons_s5', 'GSM7291242', 'GSM7291243', 'GSM7291244',
       'GSM7291245', 'GSM7291246', 'GSM7291247', 'GSM7291248',
       'GSM7291249', 'GSM7291250', 'GSM7291251', 'GSM7291252',
       'GSM7291253', 'GSM7291254', 'GSM7291255', 'GSM7291256'],
      dtype=object)

In [4]:

meta_ol = meta.copy()
# counts of cell types per sample
counts = pd.crosstab(
    meta_ol["sample_id"],
    meta_ol["Subtype"]
)

print(counts.shape)
print(counts.head())
# sample meta
sample_meta = (
    meta_ol[["sample_id", "condition"]]
    .drop_duplicates()
    .set_index("sample_id")
)

sample_meta = sample_meta.loc[counts.index]

cond_order = [
    "control",
    "MS"
]

sample_meta["condition"] = pd.Categorical(
    sample_meta["condition"],
    categories=cond_order,
    ordered=True
)

sample_meta = sample_meta.sort_values("condition")
counts = counts.loc[sample_meta.index]


# check rows match
all(counts.index == sample_meta.index)
import anndata as ad

count_adata = ad.AnnData(
    X=counts.values.astype(int),
    obs=sample_meta.copy(),
    var=pd.DataFrame(index=counts.columns)
)

count_adata.obs["condition"] = sample_meta["condition"].values
print(count_adata)
proportions = counts.div(
    counts.sum(axis=1),
    axis=0
)

proportions["condition"] = sample_meta["condition"]

print(
    proportions.groupby("condition")
    .mean()
    .T
    .to_string()
)

(32, 32)
Subtype     CD44-hi Astrocyte  COP  EAAT2-hi Astrocyte  Early Fetal OPC  \
sample_id                                                                 
GSM7291242                 28    4                   3                0   
GSM7291243                554   21                  22                0   
GSM7291244                344   14                   2                0   
GSM7291245                383   42                   3                0   
GSM7291246                375    3                   5                0   

Subtype     Early OL  Endothelial  Ependymal  Erythroblast  Fetal COP  \
sample_id                                                               
GSM7291242       179            9          0             0          0   
GSM7291243        99           26          0             0          0   
GSM7291244         6            3          0             0          0   
GSM7291245       379            7          0             0          0   
GSM7291246        32       

In [5]:
count_table = counts.reset_index()
data = dat.from_pandas(
    count_table,
    covariate_columns=["sample_id"]
)

# Attach condition to AnnData metadata
data.obs["condition"] = sample_meta.loc[
    data.obs["sample_id"].to_numpy(), "condition"
].to_numpy()


In [6]:
model = mod.CompositionalAnalysis(
    data,
    formula="condition",
    reference_cell_type="automatic"
)

result = model.sample_nuts(
    num_results=5000,
    num_burnin=2000,
    verbose=True
)

Automatic reference selection! Reference cell type set to Neuron
Zero counts encountered in data! Added a pseudocount of 0.5.


2026-10-21 00:03:53.895347: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-10-21 00:03:57.034448: I external/local_xla/xla/service/service.cc:163] XLA service 0x7f5aa40077c0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-10-21 00:03:57.034466: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
2026-10-21 00:03:57.091751: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
  0%|          | 0/5000 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1792555437.657043   17045 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
 12%|█▏        | 612/5000 [04

In [ ]:
# result table
result.summary_extended()

### ATLAS Dev-Subtype Comparison

In [ ]:
meta = pd.read_csv("../output/tables/mapping/atlas_meta.csv")
meta["dev_group"] = meta["dev.group"]

In [ ]:
meta["sample_id"].unique()

In [ ]:

meta_ol = meta.copy()
# counts of cell types per sample
counts = pd.crosstab(
    meta_ol["sample_id"],
    meta_ol["Subtype"]
)

print(counts.shape)
print(counts.head())
# sample meta
sample_meta = (
    meta_ol[["sample_id", "dev_group"]]
    .drop_duplicates()
    .set_index("sample_id")
)

sample_meta = sample_meta.loc[counts.index]

cond_order = [
    "Fetal",
    "Infant",
    "Child",
    "Adolescent",
    "Adult"
]

sample_meta["dev_group"] = pd.Categorical(
    sample_meta["dev_group"],
    categories=cond_order,
    ordered=True
)

sample_meta = sample_meta.sort_values("dev_group")
counts = counts.loc[sample_meta.index]


# check rows match
all(counts.index == sample_meta.index)
import anndata as ad

count_adata = ad.AnnData(
    X=counts.values.astype(int),
    obs=sample_meta.copy(),
    var=pd.DataFrame(index=counts.columns)
)

count_adata.obs["dev_group"] = sample_meta["dev_group"].values
print(count_adata)
proportions = counts.div(
    counts.sum(axis=1),
    axis=0
)

proportions["dev_group"] = sample_meta["dev_group"]

print(
    proportions.groupby("dev_group")
    .mean()
    .T
    .to_string()
)

In [ ]:
count_table = counts.reset_index()
data = dat.from_pandas(
    count_table,
    covariate_columns=["sample_id"]
)

# Attach condition to AnnData metadata
data.obs["dev_group"] = sample_meta.loc[
    data.obs["sample_id"].to_numpy(), "dev_group"
].to_numpy()




In [ ]:
development_score = {
    "Fetal": 0,
    "Infant": 1,
    "Child": 2,
    "Adolescent": 3,
    "Adult": 4
}

data.obs["dev_score"] = data.obs["dev_group"].map(development_score).astype(float)

In [ ]:
model_devscore =mod.CompositionalAnalysis(
    data,
    formula="dev_score"
)

result = model.sample_nuts(
    num_results=5000,
    num_burnin=2000,
    verbose=True
)

In [ ]:
# Summary table
result.summary_extended()

### ATLAS Dev-CellType Comparison

In [ ]:
meta = pd.read_csv("../atlasmeta.csv")
meta["dev_group"] = meta["dev.group"]
#meta = meta[meta["dev.group"]]

In [ ]:
meta["sample_id"].unique()

array(['10X280_2_ABCD_1', '10X346_3_ABCD_2', '10X365_4_ABCDE_2',
       '10X370_3_ABCDE_2', '10X406_5_ABCD_2', 'SC075_MW_1275_pons_s7',
       'SC076_MW_771_pons_s7', 'SC077_MW_4327_pons_S3',
       'SC079_MW_1550_pons_s7', 'SC080_MW_5309_pons_s7',
       'SC081_MW_1864_pons_sec7', 'SC082_MW_5925_pons_sec7',
       'SC084_MW_1860_pons_sec7', 'SC085_MW_1740_pons_sec7',
       'SC087_MW_4722_pons_sec7', 'SC088_MW_4332_pons_sec9',
       'SC089_MW_1499_pons_sec9', 'SC090_MW_4925_pons_sec7',
       'SC126_MW_1136_pons_s5', 'SC127_MW_1229_pons_s7',
       'SC128_MW_1538_pons_s5', 'SC129_MW_4593_pons_s5'], dtype=object)

In [ ]:

meta_ol = meta.copy()
# counts of cell types per sample
counts = pd.crosstab(
    meta_ol["sample_id"],
    meta_ol["Subtype"]
)

print(counts.shape)
print(counts.head())
# sample meta
sample_meta = (
    meta_ol[["sample_id", "dev_group"]]
    .drop_duplicates()
    .set_index("sample_id")
)

sample_meta = sample_meta.loc[counts.index]

cond_order = [
    "Fetal",
    "Infant",
    "Child",
    "Adolescent",
    "Adult"
]

sample_meta["dev_group"] = pd.Categorical(
    sample_meta["dev_group"],
    categories=cond_order,
    ordered=True
)

sample_meta = sample_meta.sort_values("dev_group")
counts = counts.loc[sample_meta.index]


# check rows match
all(counts.index == sample_meta.index)
import anndata as ad

count_adata = ad.AnnData(
    X=counts.values.astype(int),
    obs=sample_meta.copy(),
    var=pd.DataFrame(index=counts.columns)
)

count_adata.obs["dev_group"] = sample_meta["dev_group"].values
print(count_adata)
proportions = counts.div(
    counts.sum(axis=1),
    axis=0
)

proportions["dev_group"] = sample_meta["dev_group"]

print(
    proportions.groupby("dev_group")
    .mean()
    .T
    .to_string()
)

(22, 31)
Subtype           CD44 hi Astrocyte  COP  EAAT2 hi Astrocyte  Early Fetal OPC  \
sample_id                                                                       
10X280_2_ABCD_1                   0    3                   0              317   
10X346_3_ABCD_2                  14  234                   0              739   
10X365_4_ABCDE_2                  1    4                   0              132   
10X370_3_ABCDE_2                  4   25                   0              126   
10X406_5_ABCD_2                   2   14                   0              220   

Subtype           Early OL  Endothelial  Ependymal  Erythroblast  Fetal COP  \
sample_id                                                                     
10X280_2_ABCD_1         10          192        473           438          1   
10X346_3_ABCD_2         37         1871       1255           396          6   
10X365_4_ABCDE_2         5          235        264            35        578   
10X370_3_ABCDE_2         4  

In [ ]:
count_table = counts.reset_index()
data = dat.from_pandas(
    count_table,
    covariate_columns=["sample_id"]
)

# Attach condition to AnnData metadata
data.obs["dev_group"] = sample_meta.loc[
    data.obs["sample_id"].to_numpy(), "dev_group"
].to_numpy()




NameError: name 'counts' is not defined

In [ ]:
data

AnnData object with n_obs × n_vars = 22 × 31
    obs: 'sample_id', 'dev_group'

In [ ]:
development_order = [
    "Fetal",
    "Infant",
    "Child",
    "Adolescent",
    "Adult"
]

data.obs["dev_ordered"] = pd.Categorical(
    data.obs["dev_group"],
    categories=development_order,
    ordered=True
)

In [ ]:
print(data.obs["dev_ordered"].cat.categories)
print(data.obs["dev_ordered"].cat.ordered)

Index(['Fetal', 'Infant', 'Child', 'Adolescent', 'Adult'], dtype='object')
True


In [ ]:
development_score = {
    "Fetal": 0,
    "Infant": 1,
    "Child": 2,
    "Adolescent": 3,
    "Adult": 4
}

data.obs["dev_score"] = data.obs["dev_group"].map(development_score).astype(float)

In [ ]:
print(
    data.obs[["dev_group", "dev_score"]]
    .drop_duplicates()
    .sort_values("dev_score")
)

Subtype   dev_group  dev_score
0             Fetal        0.0
5            Infant        1.0
9             Child        2.0
14       Adolescent        3.0
18            Adult        4.0


In [ ]:
# check rows match
all(counts.index == sample_meta.index)

True

In [ ]:
sample_meta

,dev_group
sample_id,
10X280_2_ABCD_1,Fetal
10X346_3_ABCD_2,Fetal
10X365_4_ABCDE_2,Fetal
10X370_3_ABCDE_2,Fetal
10X406_5_ABCD_2,Fetal
SC079_MW_1550_pons_s7,Infant
SC081_MW_1864_pons_sec7,Infant
SC075_MW_1275_pons_s7,Infant
SC076_MW_771_pons_s7,Infant


In [ ]:
model_devscore =mod.CompositionalAnalysis(
    data,
    formula="dev_score"
)

Automatic reference selection! Reference cell type set to Lymphocyte
Zero counts encountered in data! Added a pseudocount of 0.5.


2026-10-12 18:32:14.244757: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [ ]:
result = model_devscore.sample_nuts(
    num_results=5000,
    num_burnin=2000,
    verbose=True
)
result.summary()

: 

In [ ]:
result.summary_extended()

Compositional Analysis summary (extended):

Data: 22 samples, 19 cell types
Reference index: 8
Formula: dev_score
Spike-and-slab threshold: 1.000

MCMC Sampling: Sampled 5000 chain states (2000 burnin samples) in 1571.639 sec. Acceptance rate: 73.7%

Intercepts:
                                 Final Parameter  HDI 3%  HDI 97%     SD  \
Cell Type                                                                  
Astrocyte                                 -0.675  -1.143   -0.191  0.258   
Early OL                                  -1.288  -1.888   -0.708  0.316   
Endothelial                               -1.008  -1.630   -0.498  0.310   
Ependymal                                 -1.327  -1.913   -0.648  0.335   
Erythroblast                              -1.543  -2.047   -1.051  0.269   
Fetal COP                                 -1.648  -2.183   -1.156  0.273   
Fetal Neuronal Progenitor                 -1.566  -2.090   -1.010  0.282   
Intermediate Proliferative Cell           -1.084  -1.